## RNN (Recurrent Neural Network)

In [1]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

## Device
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [2]:
## Load Dataset
with open("/Users/mac/Desktop/Machine Learning/DL/DB/Names/names.txt") as f:
    words=f.read().splitlines()
    
print("Total names:", len(words))
print(words[:10])

Total names: 32033
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


In [3]:
## Build Vocabulary

chars=sorted(list(set("".join(words))))

stoi={c: i+1 for i, c in enumerate(chars)}
stoi["."]=0

itos={i: c for c , i in stoi.items()}

vocab_size=len(stoi)

print(vocab_size)

27


In [4]:
## Maximum Lenght

max_length=max(len(w) for w in words)+1
print(max_length)

16


## Dataset Class

In [15]:
class NameDataset(Dataset):
    def __init__(self, words):
        self.data=[]
        
        for word in words:
            chars="." + word + "."
            
            x=[stoi[c] for c in chars[:-1]]
            y=[stoi[c] for c in chars[1:]]
            
            x+=[0]*(max_length-len(x))
            y+=[0]*(max_length-len(y))
            
            self.data.append((torch.tensor(x), torch.tensor(y)))
            
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


In [16]:
## Train / Test Split

train, test=train_test_split(
    words, test_size=0.2,
    random_state=42
)

In [17]:
## Data Loader
train_set=NameDataset(train)
test_set=NameDataset(test)

train_loader=DataLoader(
    train_set, batch_size=128, shuffle=True
)

test_loader=DataLoader(
    test_set, batch_size=128
)

In [18]:
batch = next(iter(train_loader))

print(type(batch))
print(len(batch))

for i, item in enumerate(batch):
    print(f"Item {i}:")
    print(type(item))
    print(item.shape if hasattr(item, "shape") else item)

<class 'list'>
2
Item 0:
<class 'torch.Tensor'>
torch.Size([128, 16])
Item 1:
<class 'torch.Tensor'>
torch.Size([128, 16])


## RNN Model

In [19]:
class CharRNN(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.embedding=nn.Embedding(
            vocab_size, 64
        )
        
        self.rnn=nn.RNN(
            input_size=64,
            hidden_size=128,
            num_layers=1,
            batch_first=True
        )
        
        self.dropout=nn.Dropout(0.3)
        
        self.linear=nn.Linear(
            128, vocab_size
        )
        
    def forward(self, x):
        x=self.embedding(x)
        output, hidden=self.rnn(x)
        
        output=self.dropout(output)
        
        logits=self.linear(output)
        
        return logits
    
    
    
## Create Model

model=CharRNN().to(device)

print(model)

CharRNN(
  (embedding): Embedding(27, 64)
  (rnn): RNN(64, 128, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (linear): Linear(in_features=128, out_features=27, bias=True)
)


In [26]:
### Parameters layer by layer 

total=0

for name, param in model.named_parameters():
    n=param.numel()
    total+=n
    
    print(f"{name:25} {list(param.shape)!s:20} {n}")
    
print(f"Total Parameters: {total:,}")

embedding.weight          [27, 64]             1728
rnn.weight_ih_l0          [128, 64]            8192
rnn.weight_hh_l0          [128, 128]           16384
rnn.bias_ih_l0            [128]                128
rnn.bias_hh_l0            [128]                128
linear.weight             [27, 128]            3456
linear.bias               [27]                 27
Total Parameters: 30,043


In [20]:
## Loss & Optimizer

criterion=nn.CrossEntropyLoss()

optimizer=torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)


### Train Function

In [21]:
def train():
    model.train()
    
    total_loss=0
    correct=0
    total=0
    
    for X, Y in train_loader:
        X=X.to(device)
        Y=Y.to(device)
        
        optimizer.zero_grad()
        
        logits=model(X)
        
        loss=criterion(
            logits.view(-1, vocab_size),
            Y.view(-1)
        )
        
        loss.backward()
        
        torch.nn.utils.clip_grad_norm(
            model.parameters(),
            1.0
        )
        
        optimizer.step()
        
        total_loss+=loss.item()
        
        predictions=logits.argmax(dim=2)
        
        correct+=(predictions==Y).sum().item()
        
        total+=Y.numel()
        
    accuracy=correct/total
    
    return total_loss/ len(train_loader), accuracy

### Evaluation Function

In [22]:
def evaluate():

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for X, Y in test_loader:

            X = X.to(device)
            Y = Y.to(device)

            logits = model(X)

            loss = criterion(
                logits.view(-1, vocab_size),
                Y.view(-1)
            )

            total_loss += loss.item()

            predictions = logits.argmax(dim=2)

            correct += (predictions == Y).sum().item()

            total += Y.numel()

    accuracy = correct / total

    return total_loss / len(test_loader), accuracy

### Training Loop

In [23]:
epochs=30

for epoch in range(epochs):
    train_loss, train_acc=train()
    
    test_loss, test_acc=evaluate()
    
    print(
        f"Epoch {epoch+1:2d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_acc:.4f} | "
        f"Test Loss {test_loss:.4f} | "
        f"Test Acc {test_acc:.4f}"
    )

/var/folders/08/7cs_shf56lb78lkgpnm7z4bh0000gn/T/ipykernel_61960/1827554313.py:23: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  torch.nn.utils.clip_grad_norm(


Epoch  1 | Train Loss 1.2062 | Train Acc 0.6573 | Test Loss 1.0212 | Test Acc 0.6890
Epoch  2 | Train Loss 1.0198 | Train Acc 0.6878 | Test Loss 0.9891 | Test Acc 0.6943
Epoch  3 | Train Loss 1.0002 | Train Acc 0.6919 | Test Loss 0.9789 | Test Acc 0.6971
Epoch  4 | Train Loss 0.9882 | Train Acc 0.6954 | Test Loss 0.9649 | Test Acc 0.7005
Epoch  5 | Train Loss 0.9809 | Train Acc 0.6966 | Test Loss 0.9592 | Test Acc 0.6996
Epoch  6 | Train Loss 0.9737 | Train Acc 0.6984 | Test Loss 0.9508 | Test Acc 0.7035
Epoch  7 | Train Loss 0.9692 | Train Acc 0.7001 | Test Loss 0.9486 | Test Acc 0.7049
Epoch  8 | Train Loss 0.9641 | Train Acc 0.7008 | Test Loss 0.9440 | Test Acc 0.7040
Epoch  9 | Train Loss 0.9608 | Train Acc 0.7015 | Test Loss 0.9417 | Test Acc 0.7043
Epoch 10 | Train Loss 0.9584 | Train Acc 0.7033 | Test Loss 0.9376 | Test Acc 0.7077
Epoch 11 | Train Loss 0.9547 | Train Acc 0.7038 | Test Loss 0.9352 | Test Acc 0.7078
Epoch 12 | Train Loss 0.9526 | Train Acc 0.7050 | Test Loss 0.933

## Generate Names

In [24]:
def generate_name():
    model.eval()
    
    x=torch.tensor([[0]], device=device)
    
    name=""
    
    hidden=None
    
    while True:
        emb=model.embedding(x)
        
        out, hidden=model.rnn(emb, hidden)
        
        logits=model.linear(out[:, -1])
        
        probs=F.softmax(logits, dim=1)
        
        idx=torch.multinomial(probs, 1).item()
        
        if idx==0:
            break
        
        name+=itos[idx]
        
        x=torch.tensor([[idx]], device=device)
        
    return name
        
        

In [25]:
## Test Generation

for _ in range(20):
    print(generate_name())

ailin
axanli
detzha
alani
nvina
subroenn
asham
elen
jakahio
yonda
maleahda
saana
zyriah
samella
ahaka
edbella
kencton
demenielyna
mason
mais
